### 임베딩 모델: https://github.com/nlpai-lab/KURE

In [1]:
! pip install sentence-transformers xgboost scikit-learn pandas numpy tqdm

In [2]:
import os
import re
import random
import zipfile
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import StratifiedKFold
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, precision_score, recall_score
from xgboost import XGBClassifier

import warnings
warnings.filterwarnings('ignore')

# ---------------------------
# 0) Reproducibility / Device
# ---------------------------
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(42)

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

DEVICE = get_device()
print("DEVICE =", DEVICE)


# ---------------------------
# 1) Data Locate / Load
# ---------------------------
def locate_data_dir():
    """
    train.csv / test.csv / sample_submission.csv 를 찾고,
    없으면 현재 디렉토리의 zip 파일에서 자동 추출을 시도.
    """
    cwd = Path(".").resolve()

    # 1) 현재 폴더에 바로 존재
    if (cwd / "train.csv").exists() and (cwd / "test.csv").exists():
        return cwd
    
    # 1-1) open 폴더에 존재
    if (cwd / "open" / "train.csv").exists() and (cwd / "open" / "test.csv").exists():
        return cwd / "open"

    # 2) 하위 폴더에서 탐색
    candidates = list(cwd.rglob("train.csv"))
    for tr in candidates:
        te = tr.parent / "test.csv"
        ss = tr.parent / "sample_submission.csv"
        if te.exists() and ss.exists():
            return tr.parent

    # 3) zip 자동 추출 시도
    zips = list(cwd.rglob("*.zip"))
    if len(zips) > 0:
        out_dir = cwd / "_extracted_data"
        out_dir.mkdir(parents=True, exist_ok=True)

        extracted_any = False
        for z in zips:
            try:
                with zipfile.ZipFile(z, "r") as zz:
                    names = [Path(n).name for n in zz.namelist()]
                    if ("train.csv" in names) and ("test.csv" in names) and ("sample_submission.csv" in names):
                        zz.extractall(out_dir)
                        extracted_any = True
            except Exception:
                pass

        if extracted_any:
            candidates = list(out_dir.rglob("train.csv"))
            for tr in candidates:
                te = tr.parent / "test.csv"
                ss = tr.parent / "sample_submission.csv"
                if te.exists() and ss.exists():
                    return tr.parent

    raise FileNotFoundError("train.csv / test.csv / sample_submission.csv 를 찾지 못했습니다.")

DATA_DIR = locate_data_dir()
print("DATA_DIR =", DATA_DIR)

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sub = pd.read_csv(DATA_DIR / "sample_submission.csv")

TARGET = "completed"
ID_COL = "ID"

assert TARGET in train.columns, "train에 completed 컬럼이 없습니다."
assert TARGET not in test.columns, "test에 completed 컬럼이 있으면 안 됩니다."
assert ID_COL in train.columns and ID_COL in test.columns and ID_COL in sub.columns

print("\n[EDA] Shapes")
print("train:", train.shape, "test:", test.shape, "sub:", sub.shape)

print("\n[EDA] Target distribution")
y = train[TARGET].astype(int).values
print(pd.Series(y).value_counts().sort_index())
print("pos_rate =", y.mean())


# ---------------------------
# 2) KoE5 모델 로드
# ---------------------------
print("\n[Model] Loading KoE5 embedding model...")
embedding_model = SentenceTransformer("nlpai-lab/KoE5")
print("[Model] KoE5 loaded successfully!")

c:\Users\dell\bda\.conda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DEVICE = cpu
DATA_DIR = C:\Users\dell\bda\open

[EDA] Shapes
train: (748, 46) test: (814, 45) sub: (814, 2)

[EDA] Target distribution
0    525
1    223
Name: count, dtype: int64
pos_rate = 0.29812834224598933

[Model] Loading KoE5 embedding model...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 1164.77it/s, Materializing param=pooler.dense.weight]                               


[Model] KoE5 loaded successfully!


## 데이터 전처리 수정

In [3]:
# ============================================================
# 🔧 개선된 전처리 파이프라인
# ============================================================

def advanced_preprocessing(train_df, test_df, save_path="data_process/"):
    """
    개선된 전처리 파이프라인
    
    Returns:
    --------
    train_processed, test_processed, preprocessing_info
    """
    train_proc = train_df.copy()
    test_proc = test_df.copy()
    
    # ID 보관 (제출용)
    train_id = train_proc['ID'].copy()
    test_id = test_proc['ID'].copy()
    
    print("="*60)
    print("🔧 개선된 전처리 파이프라인 시작")
    print("="*60)
    
    # ---------------------------
    # 1단계: 제거 컬럼 (ID 제외)
    # ---------------------------
    drop_columns = [
        'generation',            # 모든 값이 9로 동일
        'contest_award',         # 100% 결측치
        'idea_contest',          # 100% 결측치
        'contest_participation', # 99.2% 결측치
        'class4',                # 99.9% 결측치 (1개만 있음)
    ]
    
    existing_drop = [c for c in drop_columns if c in train_proc.columns]
    train_proc = train_proc.drop(columns=existing_drop, errors='ignore')
    test_proc = test_proc.drop(columns=existing_drop, errors='ignore')
    print(f"✅ 1단계: 제거 컬럼 - {existing_drop}")
    
    # ---------------------------
    # 2단계: 수치형 컬럼 전처리
    # ---------------------------
    # 2.1 class2, class3 결측치 → 0 (미수강 의미)
    for col in ['class2', 'class3']:
        if col in train_proc.columns:
            train_proc[col] = train_proc[col].fillna(0)
            test_proc[col] = test_proc[col].fillna(0)
    
    # 2.2 completed_semester 이상치 처리 및 결측치 대체
    if 'completed_semester' in train_proc.columns:
        # 이상치 수정 (12 초과 값 → 12로 클리핑)
        train_proc['completed_semester'] = train_proc['completed_semester'].apply(
            lambda x: min(x, 12) if pd.notna(x) and x > 12 else x
        )
        test_proc['completed_semester'] = test_proc['completed_semester'].apply(
            lambda x: min(x, 12) if pd.notna(x) and x > 12 else x
        )
        # 결측치 → 중앙값
        median_semester = train_proc['completed_semester'].median()
        train_proc['completed_semester'] = train_proc['completed_semester'].fillna(median_semester)
        test_proc['completed_semester'] = test_proc['completed_semester'].fillna(median_semester)
    
    # 2.3 major_data: bool → int
    if 'major_data' in train_proc.columns:
        train_proc['major_data'] = train_proc['major_data'].astype(int)
        test_proc['major_data'] = test_proc['major_data'].astype(int)
    
    print(f"✅ 2단계: 수치형 전처리 완료 (class2/3→0, completed_semester 이상치 처리)")
    
    # ---------------------------
    # 3단계: Binary Encoding
    # ---------------------------
    binary_mappings = {
        're_registration': {'예': 1, '아니요': 0},
        'nationality': {'내국인': 0, '외국인': 1},
        'project_type': {'팀': 1, '개인': 0},
        'incumbents_level': {'주니어 (0~3년차)': 0, '시니어 (10년차 ~)': 1},
    }
    
    for col, mapping in binary_mappings.items():
        if col in train_proc.columns:
            train_proc[col] = train_proc[col].map(mapping).fillna(0).astype(int)
            test_proc[col] = test_proc[col].map(mapping).fillna(0).astype(int)
    
    print(f"✅ 3단계: Binary Encoding - {list(binary_mappings.keys())}")
    
    # ---------------------------
    # 4단계: One-Hot Encoding
    # ---------------------------
    onehot_cols = ['major type', 'major1_1', 'major1_2', 'job']
    
    # 결측치 처리
    for col in onehot_cols:
        if col in train_proc.columns:
            train_proc[col] = train_proc[col].fillna('없음')
            test_proc[col] = test_proc[col].fillna('없음')
    
    # One-Hot Encoding (train + test 합쳐서 처리 후 분리)
    existing_onehot = [c for c in onehot_cols if c in train_proc.columns]
    
    if existing_onehot:
        # Train과 Test 구분을 위한 임시 컬럼
        train_proc['_is_train_'] = 1
        test_proc['_is_train_'] = 0
        
        combined = pd.concat([train_proc, test_proc], axis=0, ignore_index=True)
        combined = pd.get_dummies(combined, columns=existing_onehot, prefix=existing_onehot)
        
        # 다시 분리
        train_proc = combined[combined['_is_train_'] == 1].drop(columns=['_is_train_']).reset_index(drop=True)
        test_proc = combined[combined['_is_train_'] == 0].drop(columns=['_is_train_']).reset_index(drop=True)
    
    print(f"✅ 4단계: One-Hot Encoding - {existing_onehot}")
    
    # ---------------------------
    # 5단계: Class 컬럼 파생변수
    # ---------------------------
    # 5.1 수강 개수 (몇 개 수업까지 수강했는지)
    def get_num_classes(row):
        count = 1  # class1은 모두 있음
        if 'class2' in row and row['class2'] > 0:
            count = 2
        if 'class3' in row and row['class3'] > 0:
            count = 3
        return count
    
    train_proc['num_classes_taken'] = train_proc.apply(get_num_classes, axis=1)
    test_proc['num_classes_taken'] = test_proc.apply(get_num_classes, axis=1)
    
    # 5.2 복수 수강 여부
    train_proc['is_multi_class'] = (train_proc['num_classes_taken'] >= 2).astype(int)
    test_proc['is_multi_class'] = (test_proc['num_classes_taken'] >= 2).astype(int)
    
    # 5.3 과목별 Multi-Hot Encoding
    # 모든 과목 코드 추출
    all_courses = set()
    for col in ['class1', 'class2', 'class3']:
        if col in train_df.columns:
            courses = train_df[col].dropna().unique()
            all_courses.update([int(c) for c in courses if pd.notna(c) and c > 0])
    
    all_courses = sorted(all_courses)
    
    for course in all_courses:
        col_name = f'course_{course}'
        train_proc[col_name] = 0
        test_proc[col_name] = 0
        
        for class_col in ['class1', 'class2', 'class3']:
            if class_col in train_df.columns:
                train_proc.loc[train_df[class_col] == course, col_name] = 1
            if class_col in test_df.columns:
                test_proc.loc[test_df[class_col] == course, col_name] = 1
    
    # 5.4 과목 그룹 파생변수 (기초/중급/고급)
    basic_courses = [1, 2, 4, 5]
    intermediate_courses = [6, 7, 8]
    advanced_courses = [11, 12, 13]
    
    for courses, name in [(basic_courses, 'took_basic'), 
                          (intermediate_courses, 'took_intermediate'), 
                          (advanced_courses, 'took_advanced')]:
        course_cols = [f'course_{c}' for c in courses if f'course_{c}' in train_proc.columns]
        if course_cols:
            train_proc[name] = train_proc[course_cols].max(axis=1)
            test_proc[name] = test_proc[course_cols].max(axis=1)
    
    print(f"✅ 5단계: Class 파생변수 - num_classes_taken, is_multi_class, course_*, took_*")
    
    # ---------------------------
    # 6단계: 결측치 기반 파생변수
    # ---------------------------
    # 복수전공 여부
    if 'major1_2' in train_df.columns:
        train_proc['has_major2'] = (~train_df['major1_2'].isnull()).astype(int)
        test_proc['has_major2'] = (~test_df['major1_2'].isnull()).astype(int)
    
    # 재수강생 여부
    prev_cols = [col for col in train_df.columns if col.startswith('previous_class')]
    if prev_cols:
        train_proc['is_returning_student'] = (~train_df[prev_cols[0]].isnull()).astype(int)
        test_proc['is_returning_student'] = (~test_df[prev_cols[0]].isnull()).astype(int)
    
    print(f"✅ 6단계: 결측치 기반 파생변수 - has_major2, is_returning_student")
    
    # ---------------------------
    # ID 복원 및 저장
    # ---------------------------
    train_proc['ID'] = train_id.values
    test_proc['ID'] = test_id.values
    
    # 컬럼 순서 정리 (ID를 맨 앞으로)
    cols = ['ID'] + [c for c in train_proc.columns if c != 'ID']
    train_proc = train_proc[cols]
    
    cols_test = ['ID'] + [c for c in test_proc.columns if c != 'ID']
    test_proc = test_proc[cols_test]
    
    print("="*60)
    print("🔧 전처리 완료!")
    print("="*60)
    print(f"Train shape: {train_proc.shape}")
    print(f"Test shape: {test_proc.shape}")
    
    # 전처리 정보
    preprocessing_info = {
        'dropped_columns': existing_drop,
        'binary_columns': list(binary_mappings.keys()),
        'onehot_columns': existing_onehot,
        'derived_columns': ['num_classes_taken', 'is_multi_class', 'has_major2', 'is_returning_student']
    }
    
    return train_proc, test_proc, preprocessing_info


# ============================================================
# 전처리 실행
# ============================================================
print("\n" + "="*60)
print("📊 전처리 실행")
print("="*60)

# 원본 데이터 복사
train_original = train.copy()
test_original = test.copy()

# 전처리 수행
train_processed, test_processed, prep_info = advanced_preprocessing(
    train_original, 
    test_original,
    save_path="data_process/"
)

# 결과 확인
print(f"\n[전처리 전] Train: {train.shape}, Test: {test.shape}")
print(f"[전처리 후] Train: {train_processed.shape}, Test: {test_processed.shape}")
print(f"\n[새로 생성된 컬럼]")
new_cols = [c for c in train_processed.columns if c not in train.columns]
print(f"  - 개수: {len(new_cols)}")
print(f"  - 예시: {new_cols[:10]}...")



📊 전처리 실행
🔧 개선된 전처리 파이프라인 시작
✅ 1단계: 제거 컬럼 - ['generation', 'contest_award', 'idea_contest', 'contest_participation', 'class4']
✅ 2단계: 수치형 전처리 완료 (class2/3→0, completed_semester 이상치 처리)
✅ 3단계: Binary Encoding - ['re_registration', 'nationality', 'project_type', 'incumbents_level']
✅ 4단계: One-Hot Encoding - ['major type', 'major1_1', 'major1_2', 'job']
✅ 5단계: Class 파생변수 - num_classes_taken, is_multi_class, course_*, took_*
✅ 6단계: 결측치 기반 파생변수 - has_major2, is_returning_student
🔧 전처리 완료!
Train shape: (748, 525)
Test shape: (814, 525)

[전처리 전] Train: (748, 46), Test: (814, 45)
[전처리 후] Train: (748, 525), Test: (814, 525)

[새로 생성된 컬럼]
  - 개수: 488
  - 예시: ['major type_단일 전공', 'major type_단일 전공공학 (컴퓨터 공학 제외)', 'major type_복수 전공 ( 다중전공, 이중전공 포함 )', 'major type_없음', 'major1_1_(졸업)언어정보학과/(재학)컴퓨터과학과', 'major1_1_AI 자율주행시스템공학과', 'major1_1_AI&SW학부', 'major1_1_AI빅데이터융합경영학과', 'major1_1_AI빅데이터학과', 'major1_1_AI소프트웨어융합학부 컴퓨터공학과']...


In [4]:
# ============================================================
# 📁 전처리된 데이터 저장
# ============================================================

import os

OUTPUT_DIR = "data_process/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Train 데이터 저장 (타겟 포함)
train_save = train_processed.copy()
if TARGET in train.columns:
    train_save[TARGET] = train[TARGET].values

train_save_path = OUTPUT_DIR + "processed_train.csv"
train_save.to_csv(train_save_path, index=False)
print(f"✅ 전처리된 Train 데이터 저장: {train_save_path}")
print(f"   Shape: {train_save.shape}")

# Test 데이터 저장
test_save_path = OUTPUT_DIR + "processed_test.csv"
test_processed.to_csv(test_save_path, index=False)
print(f"✅ 전처리된 Test 데이터 저장: {test_save_path}")
print(f"   Shape: {test_processed.shape}")

# 전처리 정보 저장 (재현성)
import json
prep_info_path = OUTPUT_DIR + "preprocessing_info.json"
with open(prep_info_path, 'w', encoding='utf-8') as f:
    json.dump(prep_info, f, ensure_ascii=False, indent=2)
print(f"✅ 전처리 정보 저장: {prep_info_path}")

# 저장된 데이터 샘플 확인
print("\n" + "="*60)
print("📊 전처리된 데이터 샘플")
print("="*60)
print("\n[Train 데이터 (처음 5행)]")
display(train_save.head())

print("\n[컬럼 목록]")
print(train_save.columns.tolist())


✅ 전처리된 Train 데이터 저장: data_process/processed_train.csv
   Shape: (748, 525)
✅ 전처리된 Test 데이터 저장: data_process/processed_test.csv
   Shape: (814, 525)
✅ 전처리 정보 저장: data_process/preprocessing_info.json

📊 전처리된 데이터 샘플

[Train 데이터 (처음 5행)]


,ID,school1,major_data,class1,class2,class3,re_registration,nationality,inflow_route,whyBDA,...,course_7,course_8,course_11,course_12,course_13,took_basic,took_intermediate,took_advanced,has_major2,is_returning_student
0,TRAIN_000,22,0,1,4.0,0.0,0,0,에브리타임,"큰 규모인 만큼, 커리큘럼이나 운영 등 관리가 잘 될것 같아서",...,0,0,0,0,0,1,0,0,1,0
1,TRAIN_001,1,1,8,0.0,0.0,0,0,지인 추천,"큰 규모인 만큼, 커리큘럼이나 운영 등 관리가 잘 될것 같아서",...,0,1,0,0,0,0,1,0,1,0
2,TRAIN_002,27,0,7,0.0,0.0,0,0,인스타그램,"BDA 학회원만의 혜택을 누리고 싶어서(현직자 강연, 잡 페스티벌, 기업연계 공모전 등)",...,1,0,0,0,0,0,1,0,0,0
3,TRAIN_003,1,0,7,0.0,0.0,0,0,에브리타임,혼자 공부하기 어려워서,...,1,0,0,0,0,0,1,0,1,0
4,TRAIN_004,16,1,8,0.0,0.0,0,0,지인 추천,"큰 규모인 만큼, 커리큘럼이나 운영 등 관리가 잘 될것 같아서",...,0,1,0,0,0,0,1,0,1,0



[컬럼 목록]
['ID', 'school1', 'major_data', 'class1', 'class2', 'class3', 're_registration', 'nationality', 'inflow_route', 'whyBDA', 'what_to_gain', 'hope_for_group', 'previous_class_3', 'previous_class_4', 'previous_class_5', 'previous_class_6', 'previous_class_7', 'previous_class_8', 'major_field', 'desired_career_path', 'completed_semester', 'project_type', 'time_input', 'desired_job', 'certificate_acquisition', 'desired_certificate', 'desired_job_except_data', 'incumbents_level', 'incumbents_lecture', 'incumbents_company_level', 'incumbents_lecture_type', 'incumbents_lecture_scale', 'incumbents_lecture_scale_reason', 'interested_company', 'expected_domain', 'onedayclass_topic', 'completed', 'major type_단일 전공', 'major type_단일 전공공학 (컴퓨터 공학 제외)', 'major type_복수 전공 ( 다중전공, 이중전공 포함 )', 'major type_없음', 'major1_1_(졸업)언어정보학과/(재학)컴퓨터과학과', 'major1_1_AI 자율주행시스템공학과', 'major1_1_AI&SW학부', 'major1_1_AI빅데이터융합경영학과', 'major1_1_AI빅데이터학과', 'major1_1_AI소프트웨어융합학부 컴퓨터공학과', 'major1_1_AI융합학부', 'major1_1_AI융